
# Autómata Finito Determinista (DFA) — Hacer una cita al médico

**Actividad:** Laboratorio 1   
**Curso:** INFO1148    
**Objetivo 1:** Comprender y simular un AFD   
**Objetivo 2:** Resolver los ejercicios propustos para el Laboratorio 1.

Este notebook muestra un ejemplo de **Autómata Finito Determinista (DFA)**
modelando el proceso de **hacer una cita al médico**.

El objetivo previo, es que los estudiantes comprendan mediante el ejempplo cómo modelar un flujo de acciones paso a paso de forma discreta y
puedan **interactuar** con el autómata.



## Definición del DFA

### Estados del autómata
- `Inicio` → No se ha iniciado el proceso.  
- `SeleccionarEspecialidad` → Usuario elige la especialidad (cardiología).  
- `SeleccionarFecha` → Usuario selecciona la fecha de la cita.  
- `ConfirmarDatos` → El sistema muestra los datos para confirmar.  
- `CitaAgendada` → Estado final, la cita quedó registrada.

### Alfabeto (entradas posibles)
- `especialidad` → Usuario elige la especialidad.  
- `fecha` → Usuario elige fecha disponible.  
- `confirmar` → Usuario confirma la cita.  
- `cancelar` → Usuario cancela el proceso y vuelve al inicio.

### Estado inicial
- `Inicio`

### Estado(s) de aceptación
- `CitaAgendada`


In [ ]:
# @title Instalar dependencias (ejecutar una sola vez)
import os
import shutil


def ensure_graphviz_on_path():
    """Ajusta el PATH del proceso de Python para encontrar el ejecutable 'dot' en Windows."""
    candidates = [
        r"C:\Program Files\Graphviz\bin",
        r"C:\Program Files (x86)\Graphviz\bin",
        os.path.expandvars(r"%LOCALAPPDATA%\Programs\Graphviz\bin"),
        os.path.expandvars(r"%USERPROFILE%\AppData\Local\Programs\Graphviz\bin"),
    ]
    current_path = os.environ.get("PATH", "")
    for candidate in candidates:
        if os.path.isdir(candidate) and candidate not in current_path.split(os.pathsep):
            os.environ["PATH"] = candidate + os.pathsep + current_path
            current_path = os.environ["PATH"]
    return shutil.which("dot") is not None


if not ensure_graphviz_on_path():
    try:
        !pip -q install graphviz
    except Exception:
        pass
    ensure_graphviz_on_path()

try:
    import graphviz  # visualización de grafos
    print("Graphviz importado correctamente.")
except Exception:
    !pip -q install graphviz
    import graphviz
    print("Graphviz instalado e importado correctamente.")

# Aseguramos que el binario 'dot' exista y queda disponible para renderizar
if not shutil.which("dot"):
    print("Graphviz no está en PATH. Reinstala Graphviz y reinicia VS Code.")
else:
    print("Ruta de dot:", shutil.which("dot"))

try:
    import ipywidgets as widgets  # UI interactiva
except Exception:
    !pip -q install ipywidgets
    import ipywidgets as widgets

print("Dependencias listas.")


In [ ]:

# @title Definir DFA de cita médica (cardiólogo)
# -------------------------------------------------------------------
# Esta celda define una clase simple para un DFA y el DFA de ejemplo.
# Comentarios para el alumno:
# - 'states' es el conjunto de estados posibles.
# - 'alphabet' es el conjunto de entradas válidas.
# - 'transition' mapea (estado, símbolo) -> nuevo_estado.
# - 'start_state' es el estado inicial.
# - 'accept_states' es el conjunto de estados de aceptación.
#
# El DFA modela el flujo:
#   Inicio --especialidad--> SeleccionarEspecialidad
#   SeleccionarEspecialidad --fecha--> SeleccionarFecha
#   SeleccionarFecha --confirmar--> ConfirmarDatos
#   ConfirmarDatos --confirmar--> CitaAgendada (ACEPTA)
# Y en cualquier paso intermedio, 'cancelar' devuelve a Inicio.
# -------------------------------------------------------------------

from dataclasses import dataclass
from typing import Dict, Set, Tuple, List, Optional
from IPython.display import display, HTML
import graphviz

@dataclass
class DFA:
    states: Set[str]
    alphabet: Set[str]
    transition: Dict[Tuple[str, str], str]
    start_state: str
    accept_states: Set[str]

    def step(self, state: str, symbol: str) -> str:
        """Realiza una transición desde 'state' con 'symbol'."""
        key = (state, symbol)
        if key not in self.transition:
            raise ValueError(f"No hay transición definida para {key}")
        return self.transition[key]

    def simulate(self, input_symbols: List[str]):
        """Simula el DFA sobre una secuencia de símbolos."""
        current = self.start_state
        log = []
        for i, sym in enumerate(input_symbols):
            if sym not in self.alphabet:
                raise ValueError(
                    f"Símbolo inválido: {sym}. Alfabeto permitido: {sorted(self.alphabet)}"
                )
            nxt = self.step(current, sym)
            log.append((i, current, sym, nxt))
            current = nxt
        return current, log

    def accepts(self, input_symbols: List[str]) -> bool:
        final, _ = self.simulate(input_symbols)
        return final in self.accept_states

# Definición del DFA de cita médica (cardiólogo)
states = {
    "Inicio",
    "SeleccionarEspecialidad",
    "SeleccionarFecha",
    "ConfirmarDatos",
    "CitaAgendada"
}

alphabet = {"especialidad", "fecha", "confirmar", "cancelar"}

transition = {
    ("Inicio", "especialidad"): "SeleccionarEspecialidad",
    ("SeleccionarEspecialidad", "fecha"): "SeleccionarFecha",
    ("SeleccionarFecha", "confirmar"): "ConfirmarDatos",
    ("ConfirmarDatos", "confirmar"): "CitaAgendada",

    # Cancelar en pasos intermedios vuelve a Inicio
    ("SeleccionarEspecialidad", "cancelar"): "Inicio",
    ("SeleccionarFecha", "cancelar"): "Inicio",
    ("ConfirmarDatos", "cancelar"): "Inicio",
}

start_state = "Inicio"
accept_states = {"CitaAgendada"}

cita_dfa = DFA(states, alphabet, transition, start_state, accept_states)
print("DFA de cita médica creado correctamente.")


In [ ]:

# @title Funciones para visualizar y mostrar pasos
# Comentarios para el alumno:
# - 'draw_dfa' dibuja el grafo del DFA con Graphviz.
# - 'steps_table' muestra una tabla con las transiciones tomadas.

def draw_dfa(dfa: DFA, highlight_path: Optional[List[Tuple[str, str, str]]] = None,
             highlight_state: Optional[str] = None):
    dot = graphviz.Digraph(format="png")
    dot.attr(rankdir="LR")
    dot.node("", shape="point")  # nodo invisible para flecha inicial

    # Dibujar estados (doble círculo si es de aceptación)
    for s in dfa.states:
        shape = "doublecircle" if s in dfa.accept_states else "circle"
        penwidth = "3" if highlight_state == s else "1"
        dot.node(s, shape=shape, penwidth=penwidth)

    # Flecha al estado inicial
    dot.edge("", dfa.start_state)

    # Agrupar etiquetas por (src, dst)
    edges_labels = {}
    for (src, sym), dst in dfa.transition.items():
        edges_labels.setdefault((src, dst), []).append(sym)

    highlighted_pairs = set((src, dst) for (src, sym, dst) in (highlight_path or []))

    for (src, dst), syms in edges_labels.items():
        label = ", ".join(sorted(syms))
        if (src, dst) in highlighted_pairs:
            dot.edge(src, dst, label=label, color="blue", penwidth="2")
        else:
            dot.edge(src, dst, label=label)

    return dot

def steps_table(log):
    if not log:
        display(HTML("<em>No hubo transiciones.</em>"))
        return
    rows = ["<tr><th>#</th><th>Estado actual</th><th>Entrada</th><th>Nuevo estado</th></tr>"]
    for i, s, sym, nxt in log:
        rows.append(f"<tr><td>{i}</td><td>{s}</td><td>{sym}</td><td>{nxt}</td></tr>")
    html = "<table>" + "".join(rows) + "</table>"
    display(HTML(html))


In [ ]:

# @title Ejemplo de simulación
# Comentarios para el alumno:
# - La secuencia sigue el flujo correcto hasta la aceptación.
# - Cambia la secuencia (por ejemplo, agrega 'cancelar') y observa el resultado.

seq = ["especialidad", "fecha", "confirmar", "confirmar"]
final, log = cita_dfa.simulate(seq)
print("Secuencia:", seq)
print("Estado final:", final, "| ¿Cita agendada?", final in cita_dfa.accept_states)

steps_table(log)
display(
    draw_dfa(
        cita_dfa,
        highlight_path=[(s, sym, nxt) for (_, s, sym, nxt) in log],
        highlight_state=final
    )
)


In [ ]:

# @title Interfaz interactiva
# Comentarios:
# - Ingresa símbolos separados por espacios o comas.
# - El widget mostrará el camino recorrido y si se alcanzó un estado de aceptación.

import ipywidgets as widgets

help_text = widgets.HTML(
    value=(
        "<b>Instrucciones:</b> Ingresa una secuencia con símbolos separados por espacios o comas.<br>"
        "Opciones válidas: <code>especialidad</code>, <code>fecha</code>, <code>confirmar</code>, <code>cancelar</code><br>"
        "Ejemplo: <code>especialidad fecha confirmar confirmar</code>"
    )
)

txt = widgets.Text(
    value="especialidad fecha confirmar confirmar",
    placeholder="Escribe la secuencia...",
    description="Secuencia:",
    layout=widgets.Layout(width="80%")
)

btn = widgets.Button(description="Simular", button_style="primary")
out = widgets.Output()

def parse_sequence(s: str):
    s = s.replace(",", " ").strip()
    return [p for p in s.split() if p]

def on_click(_):
    out.clear_output()
    seq = parse_sequence(txt.value)
    with out:
        print("Secuencia ingresada:", seq)
        try:
            final, log = cita_dfa.simulate(seq)
            print("Estado final:", final, "| ¿Cita agendada?", final in cita_dfa.accept_states)
            steps_table(log)
            display(
                draw_dfa(
                    cita_dfa,
                    highlight_path=[(s, sym, nxt) for (_, s, sym, nxt) in log],
                    highlight_state=final
                )
            )
        except Exception as e:
            print(" Error:", e)
            print("Opciones válidas:", sorted(cita_dfa.alphabet))

btn.on_click(on_click)
display(help_text, txt, btn, out)



# Ejercicios a resolver para el Laboratorio 1

1. **Agregar médico específico (subpaso antes de la fecha)**  
   - Nuevo estado: `SeleccionarMedico`.  
   - Nueva entrada: `medico`.  
   - Flujo sugerido: `Inicio → especialidad → SeleccionarEspecialidad → medico → SeleccionarMedico → fecha → ...`

2. **Agregar paso de pago online (después de confirmar)**  
   - Nuevo estado: `Pago`.  
   - Nueva entrada: `pago`.  
   - Solo permitir `pago` desde `ConfirmarDatos`; luego `confirmar` final para pasar a `CitaAgendada`.

3. **Múltiples especialidades**  
   - Permitir elegir entre `broncopulmunar`, `radiólogo`.  
   - Manejar transiciones diferentes desde `Inicio` según la especialidad elegida.

4. **Manejo de errores**  
   - Agregar estado `Error`.  
   - Si se ingresa un símbolo inválido en cualquier paso, transicionar a `Error` e indicar cómo recuperarse al `Inicio`.

   Justifica las decisiones de diseño.


**Revise**: en plataforma carpeta de **formatos** de como responder a este Laboratorio.

In [ ]:
# Actividad 1: Agregar médico específico

from dataclasses import dataclass
from typing import Dict, Set, Tuple, List
from IPython.display import display
import graphviz

@dataclass
class DFA:
    states: Set[str]
    alphabet: Set[str]
    transition: Dict[Tuple[str, str], str]
    start_state: str
    accept_states: Set[str]

    def step(self, state: str, symbol: str) -> str:
        key = (state, symbol)
        if key not in self.transition:
            raise ValueError(f"No hay transición definida para {key}")
        return self.transition[key]

    def simulate(self, input_symbols: List[str]):
        current = self.start_state
        log = []
        for i, sym in enumerate(input_symbols):
            if sym not in self.alphabet:
                raise ValueError(f"Símbolo inválido: {sym}")
            nxt = self.step(current, sym)
            log.append((i, current, sym, nxt))
            current = nxt
        return current, log

states = {"Inicio", "SeleccionarEspecialidad", "SeleccionarMedico", "SeleccionarFecha", "ConfirmarDatos", "CitaAgendada"}
alphabet = {"especialidad", "medico", "fecha", "confirmar", "cancelar"}
transition = {
    ("Inicio", "especialidad"): "SeleccionarEspecialidad",
    ("SeleccionarEspecialidad", "medico"): "SeleccionarMedico",
    ("SeleccionarMedico", "fecha"): "SeleccionarFecha",
    ("SeleccionarFecha", "confirmar"): "ConfirmarDatos",
    ("ConfirmarDatos", "confirmar"): "CitaAgendada",
    ("SeleccionarEspecialidad", "cancelar"): "Inicio",
    ("SeleccionarMedico", "cancelar"): "Inicio",
    ("SeleccionarFecha", "cancelar"): "Inicio",
    ("ConfirmarDatos", "cancelar"): "Inicio",
}

dfa = DFA(states, alphabet, transition, "Inicio", {"CitaAgendada"})
seq = ["especialidad", "medico", "fecha", "confirmar", "confirmar"]
final, log = dfa.simulate(seq)

print("Secuencia:", seq)
print("Estado final:", final)
print("Aceptado:", final in dfa.accept_states)
print("Transiciones:")
for paso in log:
    print(paso)

# Visualización del DFA en el mismo notebook

dot = graphviz.Digraph(format="png")
dot.attr(rankdir="LR")
dot.node("", shape="point")
for s in dfa.states:
    shape = "doublecircle" if s in dfa.accept_states else "circle"
    penwidth = "3" if s == final else "1"
    dot.node(s, shape=shape, penwidth=penwidth)
dot.edge("", dfa.start_state)

sequence_edges = {(src, dst) for _, src, _, dst in log}
drawn_edges = set()
for (src, sym), dst in dfa.transition.items():
    edge_key = (src, dst)
    if edge_key in drawn_edges:
        continue
    drawn_edges.add(edge_key)
    color = "blue" if edge_key in sequence_edges else "black"
    penwidth = "2.8" if edge_key in sequence_edges else "1.1"
    style = "bold" if edge_key in sequence_edges else "solid"
    dot.edge(src, dst, label=sym, color=color, penwidth=penwidth, style=style)

display(dot)

In [ ]:
# Actividad 2: Agregar pago online
# Se agrega el estado Pago y la entrada pago, de manera que el flujo sea:
# Inicio -> especialidad -> medico -> fecha -> confirmar -> pago -> confirmar -> CitaAgendada

from dataclasses import dataclass
from typing import Dict, Set, Tuple, List
from IPython.display import display
import graphviz

@dataclass
class DFA:
    states: Set[str]
    alphabet: Set[str]
    transition: Dict[Tuple[str, str], str]
    start_state: str
    accept_states: Set[str]

    def step(self, state: str, symbol: str) -> str:
        key = (state, symbol)
        if key not in self.transition:
            raise ValueError(f"No hay transición definida para {key}")
        return self.transition[key]

    def simulate(self, input_symbols: List[str]):
        current = self.start_state
        log = []
        for i, sym in enumerate(input_symbols):
            if sym not in self.alphabet:
                raise ValueError(f"Símbolo inválido: '{sym}'")
            nxt = self.step(current, sym)
            log.append((i, current, sym, nxt))
            current = nxt
        return current, log

states = {"Inicio", "SeleccionarEspecialidad", "SeleccionarMedico", "SeleccionarFecha", "ConfirmarDatos", "Pago", "CitaAgendada"}
alphabet = {"especialidad", "medico", "fecha", "confirmar", "pago", "cancelar"}
transition = {
    ("Inicio", "especialidad"): "SeleccionarEspecialidad",
    ("SeleccionarEspecialidad", "medico"): "SeleccionarMedico",
    ("SeleccionarMedico", "fecha"): "SeleccionarFecha",
    ("SeleccionarFecha", "confirmar"): "ConfirmarDatos",
    ("ConfirmarDatos", "pago"): "Pago",
    ("Pago", "confirmar"): "CitaAgendada",
    ("SeleccionarEspecialidad", "cancelar"): "Inicio",
    ("SeleccionarMedico", "cancelar"): "Inicio",
    ("SeleccionarFecha", "cancelar"): "Inicio",
    ("ConfirmarDatos", "cancelar"): "Inicio",
    ("Pago", "cancelar"): "Inicio",
}

dfa = DFA(states, alphabet, transition, "Inicio", {"CitaAgendada"})
seq = ["especialidad", "medico", "fecha", "confirmar", "pago", "confirmar"]
final, log = dfa.simulate(seq)

print("Secuencia:", seq)
print("Estado final:", final)
print("Aceptado:", final in dfa.accept_states)
print("Transiciones:")
for paso in log:
    print(paso)

# Visualización del DFA en el mismo notebook

dot = graphviz.Digraph(format="png")
dot.attr(rankdir="LR")
dot.node("", shape="point")
for s in dfa.states:
    shape = "doublecircle" if s in dfa.accept_states else "circle"
    penwidth = "3" if s == final else "1"
    dot.node(s, shape=shape, penwidth=penwidth)
dot.edge("", dfa.start_state)

sequence_edges = {(src, dst) for _, src, _, dst in log}
drawn_edges = set()
for (src, sym), dst in dfa.transition.items():
    edge_key = (src, dst)
    if edge_key in drawn_edges:
        continue
    drawn_edges.add(edge_key)
    color = "blue" if edge_key in sequence_edges else "black"
    penwidth = "2.8" if edge_key in sequence_edges else "1.1"
    style = "bold" if edge_key in sequence_edges else "solid"
    dot.edge(src, dst, label=sym, color=color, penwidth=penwidth, style=style)

display(dot)

In [ ]:
# Actividad 3: Múltiples especialidades
# Ahora el usuario puede elegir entre dos especialidades diferentes:
# - broncopulmonar
# - radiologo
# Cada una lleva a un camino distinto desde Inicio, pero luego converge al mismo flujo de agendamiento.

from dataclasses import dataclass
from typing import Dict, Set, Tuple, List
from IPython.display import display
import graphviz

@dataclass
class DFA:
    states: Set[str]
    alphabet: Set[str]
    transition: Dict[Tuple[str, str], str]
    start_state: str
    accept_states: Set[str]

    def step(self, state: str, symbol: str) -> str:
        key = (state, symbol)
        if key not in self.transition:
            raise ValueError(f"No hay transición definida para {key}")
        return self.transition[key]

    def simulate(self, input_symbols: List[str]):
        current = self.start_state
        log = []
        for i, sym in enumerate(input_symbols):
            if sym not in self.alphabet:
                raise ValueError(f"Símbolo inválido: '{sym}'")
            nxt = self.step(current, sym)
            log.append((i, current, sym, nxt))
            current = nxt
        return current, log

states = {"Inicio", "EspecialidadBroncopulmonar", "EspecialidadRadiologo", "SeleccionarMedico", "SeleccionarFecha", "ConfirmarDatos", "Pago", "CitaAgendada"}
alphabet = {"broncopulmonar", "radiologo", "medico", "fecha", "confirmar", "pago", "cancelar"}
transition = {
    ("Inicio", "broncopulmonar"): "EspecialidadBroncopulmonar",
    ("Inicio", "radiologo"): "EspecialidadRadiologo",
    ("EspecialidadBroncopulmonar", "medico"): "SeleccionarMedico",
    ("EspecialidadRadiologo", "medico"): "SeleccionarMedico",
    ("SeleccionarMedico", "fecha"): "SeleccionarFecha",
    ("SeleccionarFecha", "confirmar"): "ConfirmarDatos",
    ("ConfirmarDatos", "pago"): "Pago",
    ("Pago", "confirmar"): "CitaAgendada",
    ("EspecialidadBroncopulmonar", "cancelar"): "Inicio",
    ("EspecialidadRadiologo", "cancelar"): "Inicio",
    ("SeleccionarMedico", "cancelar"): "Inicio",
    ("SeleccionarFecha", "cancelar"): "Inicio",
    ("ConfirmarDatos", "cancelar"): "Inicio",
    ("Pago", "cancelar"): "Inicio",
}

dfa = DFA(states, alphabet, transition, "Inicio", {"CitaAgendada"})
seq = ["broncopulmonar", "medico", "fecha", "confirmar", "pago", "confirmar"]
final, log = dfa.simulate(seq)

print("Secuencia:", seq)
print("Estado final:", final)
print("Aceptado:", final in dfa.accept_states)
print("Transiciones:")
for paso in log:
    print(paso)

# Visualización del DFA en el mismo notebook

dot = graphviz.Digraph(format="png")
dot.attr(rankdir="LR")
dot.node("", shape="point")
for s in dfa.states:
    shape = "doublecircle" if s in dfa.accept_states else "circle"
    penwidth = "3" if s == final else "1"
    dot.node(s, shape=shape, penwidth=penwidth)
dot.edge("", dfa.start_state)

sequence_edges = {(src, dst) for _, src, _, dst in log}
drawn_edges = set()
for (src, sym), dst in dfa.transition.items():
    edge_key = (src, dst)
    if edge_key in drawn_edges:
        continue
    drawn_edges.add(edge_key)
    color = "blue" if edge_key in sequence_edges else "black"
    penwidth = "2.8" if edge_key in sequence_edges else "1.1"
    style = "bold" if edge_key in sequence_edges else "solid"
    dot.edge(src, dst, label=sym, color=color, penwidth=penwidth, style=style)

display(dot)

In [ ]:
# Actividad 4: Manejo de errores con estado Error
# En este DFA total, si el usuario ingresa una acción no permitida, se va al estado Error.
# Desde allí puede recuperarse con cancelar, volviendo a Inicio para empezar de nuevo.

from dataclasses import dataclass
from typing import Dict, Set, Tuple, List
from IPython.display import display
import graphviz

@dataclass
class DFA:
    states: Set[str]
    alphabet: Set[str]
    transition: Dict[Tuple[str, str], str]
    start_state: str
    accept_states: Set[str]

    def step(self, state: str, symbol: str) -> str:
        key = (state, symbol)
        if key not in self.transition:
            raise ValueError(f"No hay transición definida para {key}")
        return self.transition[key]

    def simulate(self, input_symbols: List[str]):
        current = self.start_state
        log = []
        for i, sym in enumerate(input_symbols):
            if sym not in self.alphabet:
                raise ValueError(f"Símbolo inválido: '{sym}'")
            nxt = self.step(current, sym)
            log.append((i, current, sym, nxt))
            current = nxt
        return current, log

states_base = {"Inicio", "EspecialidadBroncopulmonar", "EspecialidadRadiologo", "SeleccionarMedico", "SeleccionarFecha", "ConfirmarDatos", "Pago", "CitaAgendada"}
alphabet = {"broncopulmonar", "radiologo", "medico", "fecha", "confirmar", "pago", "cancelar"}
transicion_base = {
    ("Inicio", "broncopulmonar"): "EspecialidadBroncopulmonar",
    ("Inicio", "radiologo"): "EspecialidadRadiologo",
    ("EspecialidadBroncopulmonar", "medico"): "SeleccionarMedico",
    ("EspecialidadRadiologo", "medico"): "SeleccionarMedico",
    ("SeleccionarMedico", "fecha"): "SeleccionarFecha",
    ("SeleccionarFecha", "confirmar"): "ConfirmarDatos",
    ("ConfirmarDatos", "pago"): "Pago",
    ("Pago", "confirmar"): "CitaAgendada",
    ("EspecialidadBroncopulmonar", "cancelar"): "Inicio",
    ("EspecialidadRadiologo", "cancelar"): "Inicio",
    ("SeleccionarMedico", "cancelar"): "Inicio",
    ("SeleccionarFecha", "cancelar"): "Inicio",
    ("ConfirmarDatos", "cancelar"): "Inicio",
    ("Pago", "cancelar"): "Inicio",
}

states_4 = states_base.union({"Error"})
transition_4 = {}
for s in states_4:
    for sym in alphabet:
        if s == "CitaAgendada":
            transition_4[(s, sym)] = "Inicio" if sym == "cancelar" else "Error"
        elif s == "Error":
            transition_4[(s, sym)] = "Inicio" if sym == "cancelar" else "Error"
        else:
            transition_4[(s, sym)] = transicion_base.get((s, sym), "Error")

dfa = DFA(states_4, alphabet, transition_4, "Inicio", {"CitaAgendada"})
seq = ["radiologo", "pago", "cancelar", "radiologo", "medico", "fecha", "confirmar", "pago", "confirmar"]
final, log = dfa.simulate(seq)

print("Secuencia:", seq)
print("Estado final:", final)
print("Aceptado:", final in dfa.accept_states)
print("Transiciones:")
for paso in log:
    print(paso)

# Visualización del DFA en el mismo notebook

dot = graphviz.Digraph(format="png")
dot.attr(rankdir="LR", splines="spline", nodesep="0.7", ranksep="1.2", pad="0.4", ordering="out")

dot.node("Inicio", shape="circle", width="0.8", fixedsize="true")
dot.node("EspecialidadBroncopulmonar", shape="circle", width="2.8", height="2.8", fixedsize="false")
dot.node("EspecialidadRadiologo", shape="circle", width="2.6", height="2.6", fixedsize="false")
dot.node("SeleccionarMedico", shape="circle", width="2.4", height="2.4", fixedsize="false")
dot.node("SeleccionarFecha", shape="circle", width="2.4", height="2.4", fixedsize="false")
dot.node("ConfirmarDatos", shape="circle", width="2.1", height="2.1", fixedsize="false")
dot.node("Pago", shape="circle", width="1.8", height="1.8", fixedsize="false")
dot.node("Error", shape="circle", style="filled", fillcolor="#f7b3b3", color="red", penwidth="2", width="2.0", height="2.0", fixedsize="false")
dot.node("CitaAgendada", shape="doublecircle", style="filled", fillcolor="#baf1bf", color="darkgreen", penwidth="2", width="2.2", height="2.2", fixedsize="false")

with dot.subgraph() as s:
    s.attr(rank="same")
    s.node("EspecialidadBroncopulmonar")
    s.node("EspecialidadRadiologo")

with dot.subgraph() as s:
    s.attr(rank="same")
    s.node("SeleccionarMedico")
    s.node("SeleccionarFecha")
    s.node("ConfirmarDatos")

with dot.subgraph() as s:
    s.attr(rank="same")
    s.node("Pago")
    s.node("Error")

sequence_edges = {(src, dst) for _, src, _, dst in log}
actual_invalid = next(((src, dst) for _, src, _, dst in log if dst == "Error"), None)

# Una sola transición invalid por cada nodo que puede llegar a Error.
invalid_sources = {src for (src, _), dst in dfa.transition.items() if dst == "Error"}
for src in sorted(invalid_sources):
    if (src, "Error") == actual_invalid:
        continue
    dot.edge(src, "Error", label="invalid", color="red", penwidth="1.1", style="dashed")

# Las transiciones que no pertenecen a la secuencia se dibujan en negro.
drawn_edges = set()
for (src, sym), dst in dfa.transition.items():
    edge_key = (src, dst)
    if dst == "Error" or edge_key in sequence_edges or edge_key in drawn_edges:
        continue
    drawn_edges.add(edge_key)

    if src == "Error" and sym == "cancelar":
        dot.edge(src, dst, label="cancelar", color="black", penwidth="1.3", style="solid")
    elif dst == "Inicio":
        dot.edge(src, dst, label="cancelar", color="black", penwidth="1.3", style="solid")
    elif src != "Error":
        dot.edge(src, dst, label=sym, color="black", penwidth="1.1", style="solid")

# Toda la secuencia real se dibuja en azul, incluyendo cancelar e invalid.
drawn_sequence_edges = set()
for _, src, sym, dst in log:
    edge_key = (src, dst)
    if edge_key in drawn_sequence_edges:
        continue
    drawn_sequence_edges.add(edge_key)

    label = "cancelar" if dst == "Inicio" or src == "Error" else ("invalid" if dst == "Error" else sym)
    dot.edge(src, dst, label=label, color="blue", penwidth="2.8", style="bold")

display(dot)

In [11]:
# @title Interfaz interactiva final (Simulador integrado con las 4 actividades)
import ipywidgets as widgets
from IPython.display import display, HTML
import graphviz

# Configuración del autómata completo (Actividades 1, 2, 3 y 4)
states_ui = {
    "Inicio",
    "EspecialidadBroncopulmonar",
    "EspecialidadRadiologo",
    "SeleccionarMedico",
    "SeleccionarFecha",
    "ConfirmarDatos",
    "Pago",
    "CitaAgendada",
    "Error"
}

alphabet_ui = {
    "broncopulmonar",
    "radiologo",
    "medico",
    "fecha",
    "confirmar",
    "pago",
    "cancelar"
}

transicion_base_ui = {
    ("Inicio", "broncopulmonar"): "EspecialidadBroncopulmonar",
    ("Inicio", "radiologo"): "EspecialidadRadiologo",
    ("EspecialidadBroncopulmonar", "medico"): "SeleccionarMedico",
    ("EspecialidadRadiologo", "medico"): "SeleccionarMedico",
    ("SeleccionarMedico", "fecha"): "SeleccionarFecha",
    ("SeleccionarFecha", "confirmar"): "ConfirmarDatos",
    ("ConfirmarDatos", "pago"): "Pago",
    ("Pago", "confirmar"): "CitaAgendada",
    ("EspecialidadBroncopulmonar", "cancelar"): "Inicio",
    ("EspecialidadRadiologo", "cancelar"): "Inicio",
    ("SeleccionarMedico", "cancelar"): "Inicio",
    ("SeleccionarFecha", "cancelar"): "Inicio",
    ("ConfirmarDatos", "cancelar"): "Inicio",
    ("Pago", "cancelar"): "Inicio",
}

transition_ui = {}
for s in states_ui:
    for sym in alphabet_ui:
        if s == "CitaAgendada":
            transition_ui[(s, sym)] = "Inicio" if sym == "cancelar" else "Error"
        elif s == "Error":
            transition_ui[(s, sym)] = "Inicio" if sym == "cancelar" else "Error"
        else:
            transition_ui[(s, sym)] = transicion_base_ui.get((s, sym), "Error")

dfa_interfaz = DFA(states_ui, alphabet_ui, transition_ui, "Inicio", {"CitaAgendada"})

def render_dfa_ui(dfa: DFA, log: list, final_state: str):
    dot = graphviz.Digraph(format="png")
    dot.attr(rankdir="LR", splines="spline", nodesep="0.7", ranksep="1.2", pad="0.4", ordering="out")

    dot.node("Inicio", shape="circle", width="0.8", fixedsize="true")
    dot.node("EspecialidadBroncopulmonar", shape="circle", width="2.8", height="2.8", fixedsize="false")
    dot.node("EspecialidadRadiologo", shape="circle", width="2.6", height="2.6", fixedsize="false")
    dot.node("SeleccionarMedico", shape="circle", width="2.4", height="2.4", fixedsize="false")
    dot.node("SeleccionarFecha", shape="circle", width="2.4", height="2.4", fixedsize="false")
    dot.node("ConfirmarDatos", shape="circle", width="2.1", height="2.1", fixedsize="false")
    dot.node("Pago", shape="circle", width="1.8", height="1.8", fixedsize="false")
    dot.node("Error", shape="circle", style="filled", fillcolor="#f7b3b3", color="red", penwidth="2", width="2.0", height="2.0", fixedsize="false")
    dot.node("CitaAgendada", shape="doublecircle", style="filled", fillcolor="#baf1bf", color="darkgreen", penwidth="2", width="2.2", height="2.2", fixedsize="false")

    with dot.subgraph() as s:
        s.attr(rank="same")
        s.node("EspecialidadBroncopulmonar")
        s.node("EspecialidadRadiologo")

    with dot.subgraph() as s:
        s.attr(rank="same")
        s.node("SeleccionarMedico")
        s.node("SeleccionarFecha")
        s.node("ConfirmarDatos")

    with dot.subgraph() as s:
        s.attr(rank="same")
        s.node("Pago")
        s.node("Error")

    sequence_edges = {(src, dst) for _, src, _, dst in log}
    actual_invalid = next(((src, dst) for _, src, _, dst in log if dst == "Error"), None)

    invalid_sources = {src for (src, _), dst in dfa.transition.items() if dst == "Error"}
    for src in sorted(invalid_sources):
        if (src, "Error") == actual_invalid:
            continue
        dot.edge(src, "Error", label="invalid", color="red", penwidth="1.1", style="dashed")

    drawn_edges = set()
    for (src, sym), dst in dfa.transition.items():
        edge_key = (src, dst)
        if dst == "Error" or edge_key in sequence_edges or edge_key in drawn_edges:
            continue
        drawn_edges.add(edge_key)

        if src == "Error" and sym == "cancelar":
            dot.edge(src, dst, label="cancelar", color="black", penwidth="1.3", style="solid")
        elif dst == "Inicio":
            dot.edge(src, dst, label="cancelar", color="black", penwidth="1.3", style="solid")
        elif src != "Error":
            dot.edge(src, dst, label=sym, color="black", penwidth="1.1", style="solid")

    drawn_sequence_edges = set()
    for _, src, sym, dst in log:
        edge_key = (src, dst)
        if edge_key in drawn_sequence_edges:
            continue
        drawn_sequence_edges.add(edge_key)

        label = "cancelar" if dst == "Inicio" or src == "Error" else ("invalid" if dst == "Error" else sym)
        dot.edge(src, dst, label=label, color="blue", penwidth="2.8", style="bold")

    return dot

# Componentes de la interfaz
help_text = widgets.HTML(
    value=(
        "<b>Instrucciones de simulación:</b><br>"
        "Ingresa la secuencia de acciones separadas por espacios o comas.<br><br>"
        "<b>Alfabeto disponible:</b> <code>broncopulmonar</code>, <code>radiologo</code>, "
        "<code>medico</code>, <code>fecha</code>, <code>confirmar</code>, <code>pago</code>, <code>cancelar</code><br><br>"
        "<b>Ejemplos de prueba:</b><br>"
        "• <i>Flujo completo (Broncopulmonar):</i> <code>broncopulmonar medico fecha confirmar pago confirmar</code><br>"
        "• <i>Flujo completo (Radiólogo):</i> <code>radiologo medico fecha confirmar pago confirmar</code><br>"
        "• <i>Con error y reinicio:</i> <code>radiologo pago cancelar radiologo medico fecha confirmar pago confirmar</code>"
    )
)

txt_seq = widgets.Text(
    value="broncopulmonar medico fecha confirmar pago confirmar",
    placeholder="Escribe los símbolos aquí...",
    description="Secuencia:",
    layout=widgets.Layout(width="80%")
)

btn_simular = widgets.Button(description="Simular DFA", button_style="primary")
out_simulacion = widgets.Output()

def parse_input(s: str):
    s = s.replace(",", " ").strip()
    return [token for token in s.split() if token]

def ejecutar_simulacion(_):
    out_simulacion.clear_output()
    secuencia = parse_input(txt_seq.value)
    
    with out_simulacion:
        print(f"Secuencia ingresada: {secuencia}")
        try:
            estado_final, historial = dfa_interfaz.simulate(secuencia)
            es_aceptado = estado_final in dfa_interfaz.accept_states
            print(f"Estado final: {estado_final} | ¿Cita Agendada?: {es_aceptado}")
            
            steps_table(historial)
            display(render_dfa_ui(dfa_interfaz, historial, estado_final))
        except Exception as err:
            print("Error durante la simulación:", err)
            print("Entradas válidas permitidas:", sorted(dfa_interfaz.alphabet))

btn_simular.on_click(ejecutar_simulacion)
display(help_text, txt_seq, btn_simular, out_simulacion)

HTML(value='<b>Instrucciones de simulación:</b><br>Ingresa la secuencia de acciones separadas por espacios o c…

Text(value='broncopulmonar medico fecha confirmar pago confirmar', description='Secuencia:', layout=Layout(wid…

Button(button_style='primary', description='Simular DFA', style=ButtonStyle())

Output()